# Document Layout Detection: YOLO vs RT-DETR

## Data Preparation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DRIVE_PATH = "/content/drive/MyDrive/doclayout-det"

In [9]:
! pip install ultralytics
! pip install onnxruntime-gpu==1.20.1 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.8 MB/s eta 0:00:00


In [4]:
import zipfile
import os

data_path = f'{DRIVE_PATH}/data_single_cat.zip'

# Define the destination directory (current working directory)
destination_dir = '.'
# name of the folder that will be created after unzipping
unzipped_folder_name = '.data'
# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Unzip the file
print(f"Unzipping {data_path} to {destination_dir}...")
with zipfile.ZipFile(data_path, 'r') as zip_ref:
    original_folder_name = zip_ref.namelist()[0].split('/')[0]
    zip_ref.extractall(destination_dir)
print("Unzipping complete!")

original_folder_path = os.path.join(destination_dir, original_folder_name)
target_folder_path = os.path.join(destination_dir, unzipped_folder_name)
os.rename(original_folder_path, target_folder_path)

Unzipping /content/drive/MyDrive/doclayout-det/data_single_cat.zip to ....
Unzipping complete!


In [5]:
folders_to_rename = ['labels_coco_multi_cat', 'images_multi_cat']

for folder_name in folders_to_rename:
    old_subfolder_path = os.path.join(target_folder_path, folder_name)
    if os.path.exists(old_subfolder_path):
        new_folder_name = folder_name.replace('_multi_cat', '')
        new_subfolder_path = os.path.join(target_folder_path, new_folder_name)
        os.rename(old_subfolder_path, new_subfolder_path)
        print(f"Renamed: {folder_name} -> {new_folder_name}")
    else:
        print(f"Warning: Folder {folder_name} not found in {target_folder_path}")

### Convert COCO to YOLO Format

In [6]:
from ultralytics.data.converter import convert_coco
import shutil
from pathlib import Path

convert_coco(
    labels_dir=".data/labels_coco/",
    save_dir=".data/converted",
    cls91to80=False,
)

converted_dir = Path(".data/converted/")
dataset_dir = Path(".data")

split_mapping = {
    "labels_coco_train": "train",
    "labels_coco_val": "val",
    "labels_coco_test": "test"
}

for json_name, yolo_split in split_mapping.items():
    src = converted_dir / "labels" / json_name
    dst = dataset_dir / "labels" / yolo_split

    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        print(f"Moving labels from {src} to {dst}...")

        files_to_move = list(src.glob("*.txt"))
        for f in files_to_move:
            shutil.move(str(f), str(dst / f.name))
    else:
        print(f"Note: Folder {src} not found.")

if converted_dir.exists():
    shutil.rmtree(converted_dir)
    print("Successfully cleaned up temporary conversion folder.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Annotations /content/.data/labels_coco/labels_coco_test.json: 100% ━━━━━━━━━━━━ 943/943 2.3Kit/s 0.4s0.0ss
Annotations /content/.data/labels_coco/labels_coco_train.json: 100% ━━━━━━━━━━━━ 4999/4999 3.6Kit/s 1.4s0.1s
Annotations /content/.data/labels_coco/labels_coco_val.json: 100% ━━━━━━━━━━━━ 943/943 4.1Kit/s 0.2s0.0s
COCO data converted successfully.
Results saved to /content/.data/converted
Moving labels from .data/converted/labels/labels_coco_train to .data/labels/train...
Moving labels from .data/converted/labels/labels_coco_val to .data/labels/val...
Moving labels from .data/converted/labels/labels_coco_test to .data/labels/test...
Successfully cleaned up temporary conversio

In [7]:
import json
from pathlib import Path

import yaml

with open(".data/labels_coco/labels_coco_train.json") as f:
    coco = json.load(f)

categories = sorted(coco["categories"], key=lambda x: x["id"])
names = {cat["id"] - 1: cat["name"] for cat in categories}

dataset = {
    "path": str(Path(".data").resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": names,
}

with open(".data/dataset.yaml", "w") as f:
    yaml.dump(dataset, f, default_flow_style=False)

## Evaluation

In [ ]:
from ultralytics import YOLO, RTDETR
import os

YOLO_WEIGHTS = "/content/drive/MyDrive/doclayout-det/yolo26l/weights/best.pt"
RTDETR_WEIGHTS = "/content/drive/MyDrive/doclayout-det/rt_detr_l-4/weights/best.pt"

def evaluate_model(model_path, model_type="YOLO"):
    print(f" Starting Evaluation for {model_type} ({model_path})")

    if not os.path.exists(model_path):
        print(f"Error: Weights not found for {model_path}!")
        return

    if model_type == "YOLO":
        model = YOLO(model_path)
    elif model_type == "RT-DETR":
        model = RTDETR(model_path)

    metrics = model.val(
        data="/content/.data/dataset.yaml",
        split="test",
        imgsz=640,
        batch=16,
        workers=0,
        device=0,
        plots=True,
        save_txt=True,
        save_conf=True
    )

    print(f"\n--- Evaluation {model_type} ---")
    print(f"mAP@50: {metrics.box.map50:.4f}")
    print(f"mAP@50-95: {metrics.box.map:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall: {metrics.box.mr:.4f}")

    print(f"\n--- Speed-Metrics ---")
    print(f"Pre-process: {metrics.speed['preprocess']:.2f} ms")
    print(f"Inference: {metrics.speed['inference']:.2f} ms")
    print(f"Post-process: {metrics.speed['postprocess']:.2f} ms")

    return metrics

yolo_metrics = evaluate_model(YOLO_WEIGHTS, model_type="YOLO")
rt_detr_metrics = evaluate_model(RTDETR_WEIGHTS, model_type="RT-DETR")